# Step 3.g — SVM training, global (EC-Earth3 grid), MIRCA2000 cropland background, three BIO variants

For each BIO variant of 1.1.d (`A_60d`: last 60 days, Ceglar et al. 2021; `B_season`: whole season, 7 variables;
`C_crop`: phase 2 per crop from the literature) and each crop, an RBF SVM is trained on ERA5 1981–2014 over the
**MIRCA2000 cropland** (training tables of 1.1.d: presences and absences where other crops are grown).

**Method** (Ceglar et al. 2021 / A. Toreti's R scripts `svm.r`, `svmTune.R`, one-pass):
- 12 (or 7) BIO variables standardised on the training data; RBF kernel (scikit-learn `SVC` = libsvm, the library of
  R `e1071`);
- classes weighted as in the R scripts (presence weight 1, absence weight = presences / absences), passed **per class**
  (`class_weight`; the R scripts' `class.weigths=` is a misspelling that `e1071::svm` ignores, so there the weights
  had no effect);
- cost and gamma chosen by stratified cross-validation (`N_FOLDS` folds) on the AUC; the AUC is computed from the
  SVM decision score, which ranks the cells exactly as the probabilities do (checked: identical AUC to 4 decimals,
  5.7 times faster); sensitivity, specificity and balanced accuracy from the predicted class;
- final model on all the training cells with the chosen cost and gamma, `probability=True` (the suitability maps of
  step 4 use `predict_proba`).

**Oversampling (Fsmote)**: the two-pass method of the R scripts creates synthetic presences, designed for presences as
a small minority (13 % / 3.5 % of arable land in Ceglar et al. 2021); over the MIRCA2000 cropland the presences are the
majority for 5 of the 7 crops (maize 82 %) and 38–47 % for rice and spring wheat. Whether it helps is test 3.

**Three tests, one after the other** (`TEST`), each on the choices of the previous ones:
1. `TEST = "anthesis"`: variants `A_60d`, `B_season`, `C_crop`, all with the ERA5 radiation (`rad_era5`), one-pass
   → `CHOSEN_VARIANT`;
2. `TEST = "radiation"` (1.1.d rerun for `CHOSEN_VARIANT` with `RAD_EST = True`): BIO03 / BIO08 (the only variables
   using the global solar radiation, costly for the GCM runs) from ERA5 (`rad_era5`, already trained in test 1), left
   out (`no_rad`) or estimated from temperature (`rad_fao56`: BIO03e / BIO08e, FAO-56 eq. 50), one-pass
   → `CHOSEN_RAD` (is the GCM radiation needed?);
3. `TEST = "method"`, on `CHOSEN_VARIANT` and `CHOSEN_RAD`, rice and potato, the same folds for all: one-pass vs the
   two-pass method of the R scripts (with the corrections of notebook 3.d: nearest neighbours, row indexing, RBF
   pre-image), both at the cost / gamma chosen in tests 1–2 and at the cost 8 / gamma 2 written in `svmFit`
   → one-pass or two-pass for the final models.

Tests 1–3: route no_threshold (`CELL_AREA_THR = None`), MIRCA2000 cropland background, no soil. Then the background:
4. no threshold vs cellareafrac (0.05 % of the cell land area, on the crop presence and on the cropland background);
5. cropland (with the threshold of test 4) vs cropland ∩ soil suitable for the crop (GAEZ, 2.g).

Outputs in `<out_root>/3_svm_train/ERA5_<route>/<variant>_<radiation>/`: `svm_model_<...>_<crop>.joblib`
(model, scaler, BIO columns, cost, gamma, CV table), `cv_<...>_<crop>.csv` and heat map; in the parent folder the
summaries `summary_svm_<test>_<...>.csv` and `method_one_vs_two_pass_<...>.csv`.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from pathlib import Path
from joblib import Parallel, delayed
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters
CLIMATE, PERIOD = "era5", ("1981", "2014")       # training climate (tables of 1.1.d)
CELL_AREA_THR = None              # None = route no_threshold (tests 1-3); 0.0005 = route cellareafrac, presence and
                                  # background = area > 0.05 % of the cell land area (test 4); same value in 1.1.d, 3.g, 1.3.a, 4.g
VARIANTS = ["A_60d", "B_season", "C_crop"]
RAD_SETS = ["rad_era5", "no_rad", "rad_fao56"]   # BIO03 / BIO08 from ERA5 ssrd | left out | estimated (FAO-56 eq. 50)
TEST = "anthesis"                # 1 "anthesis": variants A, B, C, ERA5 radiation | 2 "radiation": CHOSEN_VARIANT with
                                 # ERA5 / no / FAO-56 radiation | 3 "method": one-pass vs two-pass on both choices
CHOSEN_VARIANT = None            # set after test 1 (tests 2 and 3); 1.1.d rerun for it with RAD_EST = True
CHOSEN_RAD = None                # set after test 2 (test 3)
METHOD_CROPS = ["Rice_rf", "Potato_rf"]   # test 3: one-pass vs two-pass on these crops
if TEST == "anthesis":
    TESTS = [(v, "rad_era5") for v in VARIANTS]
elif TEST == "radiation":
    assert CHOSEN_VARIANT in VARIANTS, "set CHOSEN_VARIANT to the variant chosen by test 1"
    TESTS = [(CHOSEN_VARIANT, r) for r in RAD_SETS]
else:
    assert CHOSEN_VARIANT in VARIANTS and CHOSEN_RAD in RAD_SETS, "set CHOSEN_VARIANT and CHOSEN_RAD (tests 1 and 2)"
    TESTS = [(CHOSEN_VARIANT, CHOSEN_RAD)]
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
COST_GRID = [0.5, 1, 2, 4, 8, 16]                # as in the former notebooks (Ceglar et al.: cost 8, gamma 2)
GAMMA_GRID = [0.1, 0.5, 1, 2, 4]
N_FOLDS = 5
N_JOBS = min(12, os.cpu_count() or 1)            # cost x gamma x fold fits run in parallel
SEED = 42
R_COST, R_GAMMA = 8, 2                          # test 3: cost / gamma written in the R scripts' svmFit
OVERWRITE = False                                # False = models already trained are read, not retrained

TAG = f"{CLIMATE.upper()}_{PERIOD[0]}_{PERIOD[1]}_ecearth3"
NAME = f"{CLIMATE.upper()}_{cu.route(CELL_AREA_THR)}"
BIO_DIR = {v: cu.out_dir(CFG, "1_biovariables", NAME, v) for v in VARIANTS}
OUT0 = cu.out_dir(CFG, "3_svm_train", NAME)
OUT = {t: cu.out_dir(CFG, "3_svm_train", NAME, f"{t[0]}_{t[1]}") for t in TESTS}
print(f"{len(TESTS)} tests x {len(CROPS)} crops | grid {len(COST_GRID)} x {len(GAMMA_GRID)}, {N_FOLDS} folds, "
      f"{N_JOBS} jobs | outputs: {OUT0}")

## 1. Functions

In [ ]:
def load(variant, crop, rad="rad_era5"):
    # training table of 1.1.d (lat, lon, BIO columns, BIO03e / BIO08e, label); BIO columns of the radiation set
    df = pd.read_csv(BIO_DIR[variant] / f"training_dataset_{TAG}_{crop}.csv")
    base = [c for c in df.columns if c.startswith("BIO") and not c.endswith("e")]
    if rad == "rad_era5":
        cols = base
    elif rad == "no_rad":
        cols = [c for c in base if c not in ("BIO03", "BIO08")]
    else:
        cols = [c + "e" if c in ("BIO03", "BIO08") else c for c in base]
    return df, cols, df[cols].values.astype(float), df["label"].values.astype(int)


def class_weight(y):
    # weights of the R scripts (presence 1, absence = presences / absences), passed per class
    return {1: 1.0, 0: (y == 1).sum() / (y == 0).sum()}


def fit_one_pass(X, y, cost, gamma, probability=False):
    return SVC(kernel="rbf", C=cost, gamma=gamma, class_weight=class_weight(y), probability=probability,
               random_state=SEED).fit(X, y)


def scores(y, score, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    return {"AUC": roc_auc_score(y, score), "SENS": sens, "SPEC": spec, "BA": (sens + spec) / 2}


def one_fold(fit, X, y, tr, te, **kw):
    # standardise on the training fold, fit, score the test fold (AUC from the decision score)
    s = StandardScaler().fit(X[tr])
    m = fit(s.transform(X[tr]), y[tr], **kw)
    Xte = s.transform(X[te])
    return scores(y[te], m.decision_function(Xte), m.predict(Xte))


def folds_of(y):
    return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))


def cv_grid(X, y):
    folds = folds_of(y)
    tasks = [(c, g, k) for c in COST_GRID for g in GAMMA_GRID for k in range(N_FOLDS)]
    res = Parallel(n_jobs=N_JOBS)(delayed(one_fold)(fit_one_pass, X, y, *folds[k], cost=c, gamma=g) for c, g, k in tasks)
    per_fold = pd.DataFrame([{"cost": c, "gamma": g, "fold": k, **r} for (c, g, k), r in zip(tasks, res)])
    return per_fold.groupby(["cost", "gamma"])[["AUC", "SENS", "SPEC", "BA"]].mean().reset_index()


def heatmap(cv, best, title, path):
    pivot = cv.pivot(index="gamma", columns="cost", values="AUC")
    fig, ax = plt.subplots(figsize=(7, 3.5))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    ax.scatter([list(pivot.columns).index(best["cost"])], [list(pivot.index).index(best["gamma"])], marker="*", s=250, color="red")
    ax.set_xlabel("cost"); ax.set_ylabel("gamma"); ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, label="CV AUC"); plt.tight_layout(); plt.savefig(path, dpi=90); plt.close(fig)

## 2. Training: cross-validation of cost and gamma, final model (every variant and crop)

In [ ]:
rows = []
for variant, rad in TESTS:
    for crop in CROPS:
        out = OUT[(variant, rad)]
        mpath = out / f"svm_model_{TAG}_{crop}.joblib"
        t0 = time.time()
        if mpath.exists() and not OVERWRITE:
            b = joblib.load(mpath)
            best = b["cv"].loc[b["cv"]["AUC"].idxmax()]
            n1, n0 = b["n_class1"], b["n_class0"]
            print(f"[already trained] {variant} {rad} {crop}")
        else:
            df, cols, X, y = load(variant, crop, rad)
            cv = cv_grid(X, y)
            cv.to_csv(out / f"cv_{TAG}_{crop}.csv", index=False)
            best = cv.loc[cv["AUC"].idxmax()]
            scaler = StandardScaler().fit(X)
            model = fit_one_pass(scaler.transform(X), y, best["cost"], best["gamma"], probability=True)
            n1, n0 = int(y.sum()), int((y == 0).sum())
            joblib.dump({"model": model, "scaler": scaler, "bio_cols": cols, "best_cost": float(best["cost"]),
                         "best_gamma": float(best["gamma"]), "crop_label": crop, "variant": variant, "radiation": rad, "cv": cv,
                         "n_class1": n1, "n_class0": n0, "method": "one_pass", "climate": CLIMATE, "period": PERIOD,
                         "train_domain": f"mirca_cropland, {cu.route(CELL_AREA_THR)}", "n_folds": N_FOLDS}, mpath)
            heatmap(cv, best, f"CV AUC — {variant} {rad} {crop}", out / f"cv_heatmap_{TAG}_{crop}.png")
            print(f"{variant:8s} {rad:9s} {crop:16s} cost {best['cost']:>4} gamma {best['gamma']:>4}  AUC {best['AUC']:.3f}  "
                  f"({(time.time() - t0) / 60:.1f} min)", flush=True)
        rows.append({"variant": variant, "radiation": rad, "crop": crop, "presences": n1, "absences": n0, "cost": best["cost"],
                     "gamma": best["gamma"], "AUC": best["AUC"], "SENS": best["SENS"], "SPEC": best["SPEC"], "BA": best["BA"]})
summary = pd.DataFrame(rows).set_index(["variant", "radiation", "crop"])
summary.to_csv(OUT0 / f"summary_svm_{TEST}_{TAG}.csv")

## 3. Results of the two tests

Cross-validated AUC (and balanced accuracy, sensitivity, specificity) of the best cost / gamma, per crop. The training
cells and labels are the same everywhere; only the BIO variables change.

In [ ]:
def table(metric, level, keep, fixed):
    t = summary[metric].xs(fixed[1], level=fixed[0]).unstack(level)
    t = t[[k for k in keep if k in t.columns]].reindex(CROPS)
    t.loc["mean"] = t.mean()
    return t

if TEST in ("anthesis", "radiation"):
    title, level, keep, fixed = (("test 1, anthesis (radiation ERA5)", "variant", VARIANTS, ("radiation", "rad_era5")) if TEST == "anthesis"
                                 else (f"test 2, radiation (variant {CHOSEN_VARIANT})", "radiation", RAD_SETS, ("variant", CHOSEN_VARIANT)))
    print(title)
    for metric in ("AUC", "BA", "SENS", "SPEC"):
        print(" ", metric); display(table(metric, level, keep, fixed).round(3))
    table("AUC", level, keep, fixed).drop("mean").plot.bar(figsize=(10, 4), ylim=(0.5, 1), ylabel="CV AUC", title=title)
    plt.tight_layout(); plt.savefig(OUT0 / f"summary_svm_auc_{TEST}_{TAG}.png", dpi=90); plt.show()
else:
    display(summary.round(3))

## 4. Test 3 (`TEST = "method"`): one-pass vs the two-pass method of the R scripts

On `CHOSEN_VARIANT` and `CHOSEN_RAD`, for rice and potato (`METHOD_CROPS`), the same folds for all: one-pass and the two-pass method of
`svmTune.R` (class-weighted SVM, Fsmote synthetic presences around the presence support vectors, second SVM; with the
corrections of notebook 3.d), both at the cost / gamma chosen by the cross-validation of section 2 and at the cost 8 /
gamma 2 written in `svmFit`. The number of synthetic presences is the one of the R scripts (presence support vectors
brought up to the number of absence support vectors), so none are added when the presence support vectors are already
the more numerous: two-pass is then identical to one-pass. If two-pass is chosen, the final models are retrained
with it (to be added then).

In [ ]:
def kernel_similarity(anchor, pool, gamma):
    """
    Gaussian-type kernel similarity between one point and every point in `pool`,
    translated from svm_andrea_R's kern.dist(): exp(-gamma * ||diff||^4)
    (the *squared* squared-norm — kept exactly as in the original script, not the
    standard RBF ||diff||^2). Self-comparisons (diff sums to exactly zero) are set
    to NaN, exactly as kern.dist() does (`if(sum(x)==0){z=NA}`).
    """
    diff = anchor - pool
    row_sum = diff.sum(axis=1)
    sq_norm = np.sum(diff ** 2, axis=1)
    sim = np.exp(-gamma * sq_norm ** 2)
    sim[row_sum == 0] = np.nan
    return sim


def rbf_preimage(x_i, x_j, lam, gamma, n_iter=15, eps=1e-10):
    """
    Approximate RBF pre-image of (1-lam)*phi(x_i) + lam*phi(x_j) via the Mika et al. (1999)
    fixed-point iteration, specialised to this 2-point combination. Replaces the naive
    x[i]-fulls[j]/2 formula the original R script flagged as a placeholder ("TO BE IMPROVED
    ... solving a pre-image problem") -- see markdown cell above.
    """
    x = (1 - lam) * x_i + lam * x_j  # initial guess: input-space interpolation
    for _ in range(n_iter):
        k_i = (1 - lam) * np.exp(-gamma * np.sum((x - x_i) ** 2))
        k_j = lam * np.exp(-gamma * np.sum((x - x_j) ** 2))
        denom = k_i + k_j
        if denom < eps:
            break
        x_new = (k_i * x_i + k_j * x_j) / denom
        if np.sum((x_new - x) ** 2) < eps:
            x = x_new
            break
        x = x_new
    return x


def fsmote(x, fulls, gamma, k, n_synth, rng):
    """
    Python translation of svm_andrea_R's Fsmote(). AUDITED against the real R source
    (svm_andrea_R lines 42-51): the R script picks its 5 neighbours via 5 sequential
    which.min(kdist) calls on kern.dist()'s output. kern.dist() returns a SIMILARITY
    (exp(-gamma*norm^4), ~1 for near points, ~0 for far points, gamma>0), so a literal
    translation of which.min() selects the 5 FARTHEST points in `fulls`, not the nearest.

    Tried that literal (farthest-neighbour) version first, per the user's "trust the R
    script" instruction -- retrained all 4 crops, reran 4.a/5.a/5.b/5.c, and compared.
    Effect was concentrated on the sparsest class (Wheat-spring_rf, 4.4% prevalence, where
    Fsmote does the most work): temporal-transferability AUC dropped from ~0.64-0.70 to
    ~0.52-0.60, i.e. markedly worse, consistent with farthest-neighbour interpolation
    injecting unrealistic/noisy synthetic points for an already-thin minority class.

    Concluded (with the user) that which.min on kern.dist is very likely an unintentional
    bug in the original R script, not a deliberate design choice: the function is literally
    named "distance" (kern.dist) and used with which.min (distance semantics: nearest =
    smallest), but the formula it implements is a similarity kernel (large = near) -- a
    classic name/semantics mismatch. Fsmote is explicitly modelled on SMOTE
    (`library(smotefamily)` in the R script), and *all* SMOTE variants interpolate with
    NEAREST neighbours by construction (stay within the local minority-class manifold);
    nothing in the R script comments or justifies a farthest-neighbour departure from that.
    The empirical result (worse performance exactly where oversampling matters most) is
    consistent with an accidental inversion, not a principled choice.

    **Reverted to nearest-neighbour** (this version) per the user's decision after seeing
    both. Note: the farthest-neighbour version was run once and the resulting models/
    downstream outputs were produced and then superseded -- the pre-this-whole-detour
    outputs are backed up under `*_pre_fsmote_fix/` sibling folders (4.a/5.a/5.b/5.c
    output dirs); the models trained with the original (first) nearest-neighbour code are
    NOT recoverable (overwritten before a backup could be made, not git-tracked). Retrain
    (3, then 4.a/5.a/5.b/5.c) to regenerate nearest-neighbour-based models and outputs.
    """
    n_x = len(x)
    k = min(k, len(fulls) - 1)
    if n_synth < n_x:
        idx = rng.integers(0, n_x, size=n_synth)
        reps_per_point = np.ones(n_synth, dtype=int)
    else:
        idx = np.arange(n_x)
        reps_per_point = np.full(n_x, round(n_synth / n_x))

    synth = []
    for i, reps in zip(idx, reps_per_point):
        sim = kernel_similarity(x[i], fulls, gamma)
        neighbours = np.argsort(np.nan_to_num(sim, nan=-np.inf))[::-1][:k]  # largest similarity = nearest (reverted from R-literal farthest -- see docstring)
        chosen = rng.integers(0, len(neighbours), size=reps)
        for c in chosen:
            j = neighbours[c]
            lam = rng.uniform(0.0, 1.0)
            synth.append(rbf_preimage(x[i], fulls[j], lam, gamma))
    return np.array(synth) if synth else np.empty((0, x.shape[1]))


def fit_svm_two_pass(X, y, cost, gamma, rng, verbose=False, probability=True):
    """
    Two-pass, support-vector-focused SVM fit, translated from Andrea Toreti's R
    routines (svm_andrea_R + svmTune_andrea_R):
      1. fit a class-weighted SVM (class 1 = presence keeps weight 1, class 0 =
         background gets weight = background/presence imbalance ratio);
      2. identify the *support vectors of the presence class* from that fit;
      3. generate enough Fsmote synthetic presence points around them to match
         the number of background support vectors (nadd = n_sv0 - n_sv1);
      4. refit a second, final SVM on the original data + synthetic points.

    X, y must already be standardized / 0-1 encoded. Returns the final SVC
    (probability=True, so predict_proba is available for AUC scoring).
    """
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    imb = n_pos / n_neg
    cw = {1: 1.0, 0: imb}

    # Pass 1 — class-weighted SVM
    svm1 = SVC(kernel='rbf', C=cost, gamma=gamma, class_weight=cw, random_state=42)
    svm1.fit(X, y)

    sv_idx = svm1.support_
    sv_y = y[sv_idx]
    n_sv_pos = int((sv_y == 1).sum())
    n_sv_neg = int((sv_y == 0).sum())
    n_add = n_sv_neg - n_sv_pos   # balance presence SVs to match background SVs (svmTune_andrea_R)

    if n_add > 0 and n_sv_pos > 0:
        sv_pos = X[sv_idx][sv_y == 1]   # presence-class support vectors to oversample around
        pool_pos = X[y == 1]            # full presence-class pool for the neighbour search
        synth = fsmote(sv_pos, pool_pos, gamma, k=5, n_synth=n_add, rng=rng)
        X_aug = np.vstack([X, synth])
        y_aug = np.concatenate([y, np.ones(len(synth), dtype=int)])
        if verbose:
            print(f"  Pass 1 SVs: {n_sv_pos} presence / {n_sv_neg} background "
                  f"-> {len(synth)} synthetic presence points added")
    else:
        X_aug, y_aug = X, y
        if verbose:
            print(f"  Pass 1 SVs: {n_sv_pos} presence / {n_sv_neg} background -> no oversampling needed")

    # Pass 2 — refit on original + synthetic data. Synthetic points are all class 1,
    # so the same {1: 1.0, 0: imb} class-weight mapping already assigns them weight 1,
    # matching the R code's `class.weigths=c(myweight, rep(1,length(lv[,1])))`.
    svm2 = SVC(kernel='rbf', C=cost, gamma=gamma, class_weight=cw,
               probability=probability, random_state=42)
    svm2.fit(X_aug, y_aug)
    return svm2


print("Andrea's two-pass oversampling method defined (kernel_similarity, fsmote, fit_svm_two_pass).")

In [ ]:
def fit_two_pass_cv(X, y, cost, gamma):
    return fit_svm_two_pass(X, y, cost, gamma, np.random.default_rng(SEED), probability=False)


ref = []
for crop in (METHOD_CROPS if TEST == "method" else []):
    df, cols, X, y = load(CHOSEN_VARIANT, crop, CHOSEN_RAD)
    folds = folds_of(y)
    b = summary.loc[(CHOSEN_VARIANT, CHOSEN_RAD, crop)]
    cv_par = {"cost": b["cost"], "gamma": b["gamma"]}
    r_par = {"cost": R_COST, "gamma": R_GAMMA}
    configs = {f"one-pass, cost/gamma of the CV": (fit_one_pass, cv_par),
               f"two-pass, cost/gamma of the CV": (fit_two_pass_cv, cv_par),
               f"one-pass, cost {R_COST} gamma {R_GAMMA}": (fit_one_pass, r_par),
               f"two-pass, cost {R_COST} gamma {R_GAMMA} (R scripts)": (fit_two_pass_cv, r_par)}
    for name, (fit, kw) in configs.items():
        t0 = time.time()
        res = Parallel(n_jobs=min(N_JOBS, N_FOLDS))(delayed(one_fold)(fit, X, y, tr, te, **kw) for tr, te in folds)
        ref.append({"crop": crop, "presence share": round(y.mean(), 2), "method": name, **pd.DataFrame(res).mean().to_dict(),
                    "minutes": round((time.time() - t0) / 60, 1)})
        print(f"{crop} | {name}: AUC {ref[-1]['AUC']:.3f} ({ref[-1]['minutes']} min)", flush=True)
if ref:
    ref = pd.DataFrame(ref).set_index(["crop", "method"])
    ref.to_csv(OUT0 / f"method_one_vs_two_pass_{CHOSEN_VARIANT}_{CHOSEN_RAD}_{TAG}.csv")
    with pd.option_context("display.max_rows", 100, "display.width", 200):
        display(ref.round(3))
        print("mean over the crops"); display(ref.groupby("method")[["AUC", "SENS", "SPEC", "BA"]].mean().round(3))